# CircuitSight — Evaluation (pull dataset + model from Google Drive)

Eval-only notebook: it **loads the dataset and the fine-tuned model from your Google Drive**, then runs the base-vs-tuned harness (§9) and the tool-offload comparison (§9d). No training cells.

**Run order (top to bottom):**
1. **Setup** — run the install cell, then **Runtime → Restart session**, then run the imports cell.
2. **Config** — defines `CFG` + `INSTRUCTION`.
3. **Load dataset (Drive)** — set `DATASET_ZIP` if your path differs.
4. **Load model (Drive)** — set `MODEL_ZIP` if your path differs.
5. **Eval harness** → **§9** (base vs tuned) → **§9d** (tool-offload).

GPU: a T4 is fine for §9d; use an **A100** for §9 (it runs 2×`EVAL_N` generations).

## 1. Setup

In [ ]:
# Install a torch version Unsloth supports (let pip choose the right CUDA build).
!pip install -q "torch==2.6.0" "torchvision==0.21.0"
!pip install -q unsloth sympy
print("installed — RESTART SESSION before importing")

In [ ]:
import torch; print("torch", torch.__version__)
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("unsloth ready")

## 2. Config

In [ ]:
CFG = dict(
    MODEL       = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",  # fallback: SmolVLM / a 2B if VRAM is tight
    MAX_IMAGE_PX= 512,     # longest image side; the main VRAM + speed knob. 512 ~= 2x faster than 768 on a T4.
    LORA_R      = 16, LORA_ALPHA = 16,
    BATCH       = 2, GRAD_ACCUM = 4,   # effective batch = BATCH*GRAD_ACCUM = 8. Bigger BATCH = faster but more VRAM; run the §4 smoke test first. OOM? -> BATCH=1, GRAD_ACCUM=8 (same effective 8, no extra VRAM). A100: BATCH 4-8 ok.
    MAX_STEPS   = 350,     # <-- caps optimizer steps (~30-40 min on a T4). Set to None to train a full epoch.
    EPOCHS      = 1,       # only used when MAX_STEPS is None
    LR          = 2e-4,
    MAX_LEN     = 2048,
    SEED        = 3407,
    DATA_DIR    = "circuitsight_dataset",   # unzipped dataset folder
    SAVE_TO_DRIVE = False,   # if True, snapshot the trained adapter to DRIVE_DIR/models (keep 2 most recent)
    DRIVE_DIR   = "/content/drive/MyDrive/CircuitSight",   # your project folder in Google Drive
    OUT_DIR     = "circuitsight_qlora",
)
# Why this many steps? EPOCHS=1 over ~10k images at effective batch BATCH*GRAD_ACCUM=4 is ~2,600
# optimizer steps (~6h on a T4) -- that is steps, not epochs. This narrow, structured behavior is
# learned in a few hundred steps, so we cap with MAX_STEPS. 350 steps ~= 2,800 images seen (effective batch 8); the
# trainer shuffles the full dataset, so all families + value-modes (numeric/symbolic/mixed) are
# sampled in proportion to the dataset mix (see CONFIG fractions). Watch the section-9 eval curve;
# raise MAX_STEPS (or set None) only if it's still climbing.
INSTRUCTION = ("You are a circuit analysis tutor. Look at the schematic and answer the question. "
    "First list every component, each with the image region it occupies as "
    "<box>[x0,y0,x1,y1]</box> in 0-1000 normalized coordinates. Then state the concepts used and "
    "the topology (what is in series/parallel). If there is a capacitor or inductor, apply its "
    "steady-state / t=0 behavior (a capacitor is open at steady state and a wire at t=0; an "
    "inductor is the reverse). Component values may be numbers (e.g. 100Ω, 12V) or symbols (e.g. "
    "R1, R2, V) — if they are symbols, give the answer as an algebraic expression in those symbols. "
    "Solve step by step, showing intermediate values (e.g. R_eq and each branch current), and "
    "end with a self-check. If a component value is not legible, say so and report the answer as "
    "null instead of guessing. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'. "
    "Then, if the circuit is a resistor network with legible values, add ONE more line giving the answer "
    "as a FORMULA in the component labels plus the value you read for each (so a calculator can do the "
    "arithmetic): 'PLAN: {\"quantity\":..., \"target_id\":..., \"expression\":\"<formula, e.g. V/(R1+R2)>\", "
    "\"values\":{\"R1\":<number>, ...}, \"unit\":...}'.")
CFG

## 3. Load the dataset (from Google Drive)

Set `DATASET_ZIP` to your dataset zip in `MyDrive/slm_project/data/`.

In [ ]:
import os, json, zipfile
from collections import Counter

# ---- Load the dataset from Google Drive ----
DATASET_ZIP = "/content/drive/MyDrive/slm_project/data/circuitsight_dataset_20260710_1.zip"  # <- your zip

# mount Drive if it isn't already
if DATASET_ZIP.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")

# sanity: the file exists and is a real (non-truncated) zip -> clear message instead of a raw BadZipFile
assert os.path.exists(DATASET_ZIP), f"{DATASET_ZIP} not found - check the path and that Drive is mounted."
with open(DATASET_ZIP, "rb") as _f:
    assert _f.read(4) == b"PK\x03\x04", f"{DATASET_ZIP} is not a valid .zip (truncated upload or wrong file)."

# the dataset was zipped from INSIDE the dataset folder, so its contents (images/, train.jsonl, ...)
# sit at the zip root -> extract into CFG["DATA_DIR"].
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(CFG["DATA_DIR"])
print("unzipped", DATASET_ZIP, "->", CFG["DATA_DIR"])

IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")
assert os.path.isdir(IMG_DIR), f"expected {IMG_DIR} after unzip - check the zip's internal structure"

def load_jsonl(name):
    p = os.path.join(CFG["DATA_DIR"], name)
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

train_rows = load_jsonl("train.jsonl")
val_rows   = load_jsonl("val_synthetic.jsonl")
print(f"train: {len(train_rows)}  val: {len(val_rows)}")
print("example keys:", list(train_rows[0].keys()))
print("value_mode mix:", Counter(r.get("value_mode","MISSING->stale zip!") for r in train_rows))
# sanity: confirm this is the PLAN-retrain dataset (resistor targets should carry a PLAN line)
_has_plan = sum("\nPLAN:" in r.get("target_output","") for r in train_rows)
print(f"targets with a PLAN line: {_has_plan}/{len(train_rows)}  (0 => generated with INCLUDE_PLAN=False)")

# --- alternative: upload the zip to /content instead of Drive ---
# from google.colab import files; up = files.upload(); DATASET_ZIP = "/content/"+next(iter(up))


## 4. Load the model to evaluate (from Google Drive)

Set `MODEL_ZIP` to your model zip in `MyDrive/slm_project/models/`. Defines `model`, `tokenizer`, `load_image`, `solve_image` (896-token cap).

In [ ]:
# ---- Load the model to evaluate, from Google Drive (run before §9 / §9d in a fresh runtime) ----
# Skip if you just trained in this session (the model is already in memory).
MODEL_ZIP = "/content/drive/MyDrive/slm_project/models/circuitsight_qlora_final_20260710_1.zip"  # <- your model zip

import os, glob, zipfile
from unsloth import FastVisionModel
from PIL import Image

# mount Drive if needed
if MODEL_ZIP.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")

# locate + integrity-check the zip (clear message instead of a raw BadZipFile)
if not os.path.exists(MODEL_ZIP):
    print("Not found. Zips in that folder:")
    for z in sorted(glob.glob(os.path.join(os.path.dirname(MODEL_ZIP), "*.zip"))): print("  ", z)
    raise FileNotFoundError(MODEL_ZIP)
with open(MODEL_ZIP, "rb") as _f:
    assert _f.read(4) == b"PK\x03\x04", f"{MODEL_ZIP} is not a valid .zip (truncated / wrong file)."

# unzip + load the adapter (finds adapter_config.json at any depth)
UNZIP = "/content/loaded_model"; os.makedirs(UNZIP, exist_ok=True)
with zipfile.ZipFile(MODEL_ZIP) as z: z.extractall(UNZIP)
ADAPTER_DIR = os.path.dirname(glob.glob(os.path.join(UNZIP, "**", "adapter_config.json"), recursive=True)[0])
print("adapter at:", ADAPTER_DIR)
model, tokenizer = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                                   use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(model); print("loaded model from", os.path.basename(MODEL_ZIP))

# image loader + solver (so §9 / §9d run even if §4 and §8 weren't run this session)
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB"); m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=896):
    FastVisionModel.for_inference(model)
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()
print("model, tokenizer, load_image, solve_image ready -> run the §9 harness, then §9 / §9d")


## 5. Evaluation harness

Defines the scorers (`parse_final` is the fixed non-greedy version), `aggregate`, and `evaluate`.

In [ ]:
import re, json, random
import sympy as smp
# ============================================================================
# Eval harness for the generalized schema:
#   FINAL: {quantity, target_id, value, unit, abstain}
# Scores, separately, the axes the BrainLift cares about:
#   component identification (count + TYPE: R vs C vs L), grounding (IoU>=0.5) +
#   grounding-hallucination, step-verified solve (final AND R_eq intermediate),
#   fabrication vs honest-abstention, concept-hallucination, and — the headline
#   "spatial blindness" number — reactive component-type accuracy (cap vs inductor).
# ============================================================================
def parse_final(text):
    m = re.search(r"FINAL:\s*(\{.*?\})", text)   # non-greedy, no DOTALL: a trailing PLAN line must NOT be swallowed
    if m:
        try:
            j = json.loads(m.group(1))
            return {"quantity":j.get("quantity"),"target_id":j.get("target_id"),
                    "value":j.get("value"),"unit":j.get("unit"),"abstain":bool(j.get("abstain",False))}
        except Exception: pass
    low = text.lower()                                     # prose fallback
    ab = ("null" in low) or ("not legible" in low) or ("cannot" in low)
    val=None
    m2 = re.search(r"answer[^=]*=\s*[^=]*?(-?\d+(?:\.\d+)?(?:e-?\d+)?)", low)
    if m2:
        try: val=float(m2.group(1))
        except Exception: val=None
    return {"quantity":None,"target_id":None,"value":val,"unit":None,"abstain":ab}

def parse_components(text):
    res_ids = set(re.findall(r"\bR(\d+)\b", text))
    has_cap = bool(re.search(r"capacitor|\u00b5F|uF|\bC1\b", text, re.I))
    has_ind = bool(re.search(r"inductor|\bmH\b|\bL1\b", text, re.I))
    m = re.search(r"(\d+)\s*resistor", text.lower())
    return {"n_res": int(m.group(1)) if m else len(res_ids), "has_cap":has_cap, "has_ind":has_ind}

def parse_boxes(text):
    pat=r"(V\d+|R\w+|C\w+|L\w+|SW|VM)\s*<box>\s*\[?\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*\]?\s*</box>"
    return {cid:[int(a),int(b),int(c),int(d)] for cid,a,b,c,d in re.findall(pat, text)}

def _iou(p,g):
    ix0,iy0,ix1,iy1=max(p[0],g[0]),max(p[1],g[1]),min(p[2],g[2]),min(p[3],g[3])
    inter=max(0,ix1-ix0)*max(0,iy1-iy0)
    u=max(0,p[2]-p[0])*max(0,p[3]-p[1])+max(0,g[2]-g[0])*max(0,g[3]-g[1])-inter
    return inter/u if u>0 else 0.0

_CKEYS={"ohm":"ohm","parallel":"parallel","series":"series","capacitor":"capacitor","inductor":"inductor"}
def _concepts_text(text):
    m=re.search(r"concepts used:\s*(.+)", text.lower())
    if not m: return None
    chunk=m.group(1).split("\n")[0]
    return {v for k,v in _CKEYS.items() if k in chunk}
def _concepts_gold(concepts):
    s=" ".join(concepts).lower(); return {v for k,v in _CKEYS.items() if k in s}

_NUM=re.compile(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?$")
_SYM_RNG=random.Random(12345)   # fixed seed -> reproducible symbolic-equivalence checks
def _is_num(x):
    return isinstance(x,(int,float)) or (isinstance(x,str) and bool(_NUM.match(x.strip())))
def sym_equal(a, b, trials=6):
    """Algebraic equivalence via random substitution: plug the same random reals into the
    shared free symbols and compare (robust to any equivalent form, e.g. R1+R2 == R2+R1)."""
    try:
        ea=smp.sympify(str(a)); eb=smp.sympify(str(b))
    except Exception:
        return str(a).replace(" ","")==str(b).replace(" ","")
    syms=sorted(ea.free_symbols | eb.free_symbols, key=str)
    if not syms:
        try: return abs(float(ea)-float(eb))<=1e-6*max(1.0,abs(float(eb)))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
    for _ in range(trials):
        subsd={s:_SYM_RNG.uniform(1.5,9.5) for s in syms}
        try: fa=float(ea.subs(subsd)); fb=float(eb.subs(subsd))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
        if abs(fa-fb) > 1e-6*max(1.0,abs(fb)): return False
    return True
def _close(pred, gold, rel=0.03):
    """Numeric (3% tol) when both numeric; else algebraic equivalence (symbolic answers)."""
    if pred is None or gold is None: return False
    if _is_num(pred) and _is_num(gold):
        return abs(float(pred)-float(gold)) <= max(1e-9, abs(float(gold))*rel)
    return sym_equal(pred, gold)

def score_record(gold, model_text, rel_tol=0.03, ground_mode="id"):
    ga=bool(gold["abstain"]); fam=gold.get("family","dc_resistor")
    p=parse_final(model_text); comp=parse_components(model_text); mb=parse_boxes(model_text)
    gc=gold["gold_components"]
    res={"family":fam,"question_type":gold.get("question_type"),"value_mode":gold.get("value_mode","numeric"),
         "abstain_gold":ga,"abstain_pred":bool(p["abstain"]),
         "comp_ok":None,"react_type_ok":None,"req_ok":None,"answer_ok":None,"intermediates_ok":None,
         "fabricated":None,"honest_abstain":None,
         "concept_declared":None,"concept_hallucinated":None,
         "grounding_ok":None,"grounding_hallucinated":None}
    # component identification: right resistor COUNT and right presence of cap/inductor TYPE
    res["comp_ok"]=(comp["n_res"]==gc.get("resistor")) and \
                   (comp["has_cap"]==("capacitor" in gc)) and (comp["has_ind"]==("inductor" in gc))
    if fam=="reactive":                                    # cap-vs-inductor confusion (headline)
        want_cap="capacitor" in gc
        res["react_type_ok"]=bool((comp["has_cap"] and not comp["has_ind"]) if want_cap
                                  else (comp["has_ind"] and not comp["has_cap"]))
    # grounding: fraction of gold components boxed with IoU>=0.5; hallucination = boxes that miss
    gb=gold.get("gold_boxes",{})   # ground_mode="iou": id-agnostic greedy match (real images number their own way)
    if gb:
        if ground_mode=="iou":
            preds=list(mb.values()); used=[False]*len(preds); hit=0
            for g in gb.values():
                best=0.0; bi=-1
                for i,pbx in enumerate(preds):
                    if used[i]: continue
                    v=_iou(pbx,g)
                    if v>best: best=v; bi=i
                if bi>=0 and best>=0.5: used[bi]=True; hit+=1
            res["grounding_ok"]=hit/len(gb)
            if preds: res["grounding_hallucinated"]=sum(1 for u in used if not u)/len(preds)
        else:
            res["grounding_ok"]=sum(1 for cid,g in gb.items() if cid in mb and _iou(mb[cid],g)>=0.5)/len(gb)
            if mb:
                res["grounding_hallucinated"]=sum(1 for cid,m in mb.items()
                                                  if cid not in gb or _iou(m,gb[cid])<0.5)/len(mb)
    # concept scoping
    gset=_concepts_gold(gold.get("concepts",[])); pc=_concepts_text(model_text)
    if pc is None: res["concept_declared"]=False
    else: res["concept_declared"]=True; res["concept_hallucinated"]=len(pc-gset)>0
    # answer / abstention
    if ga:
        gave=(p["value"] is not None) and (not p["abstain"])
        res["fabricated"]=gave; res["honest_abstain"]=bool(p["abstain"]) and not gave
    else:
        ga_dict=gold.get("gold_answer") or {}          # perception-only records have gold_answer=None
        gv=ga_dict.get("value")                        # float | expr-string | None
        gvals=gold.get("gold_values") or {}
        res["answer_ok"]=_close(p["value"], gv, rel_tol) if gv is not None else None
        qt=gold.get("question_type")
        if qt=="resistance":
            res["req_ok"]=res["answer_ok"]
        elif qt in ("topology","components") or gv is None:   # perception-only: no numeric intermediate
            res["req_ok"]=None
        elif gvals.get("I_total")==0:                  # open circuit: no loop-R_eq intermediate
            res["req_ok"]=None
        elif "R_eq" in gvals:
            gold_req=gvals["R_eq"]
            if _is_num(gold_req):
                mR=re.search(r"r_eq\s*=\s*(-?\d+\.?\d*(?:[eE][+-]?\d+)?)", model_text, re.I)
                predR=mR.group(1) if mR else None
            else:
                mR=re.search(r"r_eq\s*=\s*([^\n]+?)\s*(?:ohm|\u03a9|\(=|$)", model_text, re.I) or \
                   re.search(r"r_eq\s*=\s*([^\n.]+)", model_text, re.I)
                predR=mR.group(1).strip().rstrip(".") if mR else None
            res["req_ok"]=_close(predR, gold_req, rel_tol)
        gbc=gvals.get("branch_currents",{})            # full intermediate verification
        if gbc:
            pbc={cid:v.strip() for cid,v in re.findall(r"I\((R\w+)\)\s*=\s*([^,\n]+?)\s*A", model_text)}
            checked=ok=0
            for cid,gvv in gbc.items():
                if _is_num(gvv) and abs(float(gvv))<1e-3: continue   # skip sub-mA (numeric rounding noise)
                checked+=1; pv=pbc.get(cid)
                if pv is not None and _close(pv, gvv, rel_tol): ok+=1
            res["intermediates_ok"]=(ok==checked) if checked else None
    return res

def aggregate(results):
    non=[r for r in results if not r["abstain_gold"]]; ab=[r for r in results if r["abstain_gold"]]
    rj =[r for r in results if r["family"]=="reactive"]
    def frac(xs,k):
        xs=[r[k] for r in xs if r[k] is not None]; return round(sum(xs)/len(xs),4) if xs else None
    step=[r for r in non if r["comp_ok"] and (r["req_ok"] in (True,None)) and (r["intermediates_ok"] in (True,None)) and r["answer_ok"]]
    qts=sorted(set(r["question_type"] for r in non if r["question_type"]))
    _gr=frac(results,"grounding_ok"); _gh=frac(results,"grounding_hallucinated")
    _gp=(round(1-_gh,4) if _gh is not None else None)
    _gf1=(round(2*_gp*_gr/(_gp+_gr),4) if (_gp and _gr and _gp+_gr>0) else None)
    _tp=sum(1 for r in results if r["abstain_gold"] and r["abstain_pred"])
    _fp=sum(1 for r in results if (not r["abstain_gold"]) and r["abstain_pred"])
    _fn=sum(1 for r in results if r["abstain_gold"] and not r["abstain_pred"])
    _apr=(round(_tp/(_tp+_fp),4) if _tp+_fp else None); _arc=(round(_tp/(_tp+_fn),4) if _tp+_fn else None)
    return {"n_total":len(results),"n_abstain":len(ab),"n_reactive":len(rj),
            "component_accuracy":frac(results,"comp_ok"),
            "reactive_type_accuracy":frac(rj,"react_type_ok"),
            "Req_accuracy":frac(non,"req_ok"),"answer_accuracy":frac(non,"answer_ok"),
            "intermediates_accuracy":frac(non,"intermediates_ok"),
            "step_verified_accuracy":round(len(step)/len(non),4) if non else None,
            "fabrication_rate":frac(ab,"fabricated"),"honest_abstention_rate":frac(ab,"honest_abstain"),
            "grounding_accuracy":frac(results,"grounding_ok"),
            "grounding_hallucination_rate":frac(results,"grounding_hallucinated"),
            "concept_declared_rate":frac(results,"concept_declared"),
            "concept_hallucination_rate":frac(results,"concept_hallucinated"),
            "grounding_recall":_gr,"grounding_precision":_gp,"grounding_f1":_gf1,
            "abstention_precision":_apr,"abstention_recall":_arc,
            "answer_accuracy_by_value_mode":{vm:frac([r for r in non if r.get("value_mode")==vm],"answer_ok") for vm in ("numeric","symbolic","mixed")},
            "answer_accuracy_by_qtype":{q:frac([r for r in non if r["question_type"]==q],"answer_ok") for q in qts}}

def evaluate(model, tokenizer, rows, n=None, ground_mode="id"):
    # ground_mode="iou" for the real-world set (model numbers components its own way)
    rows = rows[:n] if n else rows
    return aggregate([score_record(r, solve_image(load_image(r["image"]), r["question"], model, tokenizer),
                                    ground_mode=ground_mode)
                      for r in rows])
print("eval harness loaded (generalized schema + grounding + component-type confusion)")

## §9 — base vs. tuned

Reloads the base model and compares it to the tuned model on the held-out val set. `EVAL_N=30` (≈60 generations). Use an A100.

In [ ]:
# Base vs tuned on the held-out synthetic val set (use the real-world eval set for the headline number).
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N = min(30, len(EVAL_ROWS))   # each unit = 2 gens (base+tuned), up to 896 tok: ~15s/gen A100, ~60s T4. Lower for a quick pass; raise for firmer numbers.

# --- tuned (current, fine-tuned model) ---
tuned_metrics = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N)
print("TUNED :", tuned_metrics)

# --- base (reload a fresh, un-tuned model) ---
base_model, base_tok = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(base_model)
base_metrics = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N)
print("BASE  :", base_metrics)

print("\n=== DELTA (tuned - base) ===")
for k in tuned_metrics:
    if isinstance(tuned_metrics[k],(int,float)) and isinstance(base_metrics.get(k),(int,float)):
        print(f"{k:28s}: {base_metrics[k]:.3f} -> {tuned_metrics[k]:.3f}  ({tuned_metrics[k]-base_metrics[k]:+.3f})")

# per-question-type answer accuracy (where the tuned model helps most)
print("\n--- answer accuracy by question type (base -> tuned) ---")
bt = base_metrics.get("answer_accuracy_by_qtype",{}); tt = tuned_metrics.get("answer_accuracy_by_qtype",{})
for q in sorted(set(bt)|set(tt)):
    b,t = bt.get(q), tt.get(q)
    bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
    print(f"  {q:12s}: {bs} -> {ts}")
print("\n--- answer accuracy by value mode (base -> tuned): does it reason symbolically even when arithmetic slips? ---")
bv = base_metrics.get("answer_accuracy_by_value_mode",{}); tv = tuned_metrics.get("answer_accuracy_by_value_mode",{})
for vm in ("numeric","symbolic","mixed"):
    b,t = bv.get(vm), tv.get(vm)
    bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
    print(f"  {vm:9s}: {bs} -> {ts}")

### §9d — Tool-offloaded solving (model sets up, calculator computes)

The value-mode split showed the fine-tuned model's weakness is **arithmetic**, not circuit reasoning (numeric 0% vs symbolic ~31%). With `INCLUDE_PLAN=True` in the dataset, the model is trained to end each resistor-network answer with a machine-readable `PLAN` line — the answer as a **formula** in the component labels plus the **values it read** (e.g. `PLAN: {"expression":"V/(R1+R2)","values":{"V":12,"R1":220,"R2":330}}`). A deterministic **sympy** step substitutes the values and computes the number, so the model never does the arithmetic.

This cell scores **numeric resistor problems** from a single generation: the model's own `FINAL` value (raw arithmetic) vs the `PLAN`→sympy value (tool-offloaded), head-to-head. The tool is **safe** — only component labels that were actually read may appear as variables; functions, code, or missing readings abstain.

**Requires the PLAN-retrained model** (regenerate the dataset with `INCLUDE_PLAN=True`, then retrain). On a model trained without it, no `PLAN` is emitted and the cell says so. Prereqs: a loaded model, the §9 harness (`_close`, `parse_final`, `solve_image`, `load_image`), and `val_rows` (§3).


In [ ]:
# ---- §9d: tool-offloaded solving — PLAN-trained model emits a formula + readings; sympy computes ----
import re, json
import sympy as smp

_PLAN=re.compile(r"PLAN:\s*(\{.*\})", re.DOTALL)
_SAFE=re.compile(r"^[0-9A-Za-z_+\-*/(). \t]+$")     # no quotes/brackets/commas/semicolons
_IDENT=re.compile(r"[A-Za-z_][A-Za-z_0-9]*")

def parse_plan(text):
    m=_PLAN.search(text)
    if not m: return None
    blob=m.group(1)
    try: return json.loads(blob)
    except Exception:
        depth=0; end=None
        for i,ch in enumerate(blob):
            if ch=="{": depth+=1
            elif ch=="}":
                depth-=1
                if depth==0: end=i+1; break
        return json.loads(blob[:end]) if end else None

def _num(x):
    if isinstance(x,(int,float)): return float(x)
    if isinstance(x,str):
        m=re.search(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?", x.replace(",",""))
        return float(m.group(0)) if m else None
    return None

def tool_compute(plan):
    """Substitute read values into the model's formula and evaluate with sympy. Safe: only component
    labels with a read value may appear; functions / code / missing readings -> abstain."""
    base={"quantity":None,"target_id":None,"value":None,"unit":None,"abstain":True}
    if not plan: return base
    for k in ("quantity","target_id","unit"):
        if plan.get(k) is not None: base[k]=plan.get(k)
    expr=str(plan.get("expression","")).strip().rstrip(".")
    expr=expr.replace("Ω","").replace("×","*").replace("·","*").replace("÷","/").replace("^","**")
    expr=re.sub(r"([A-Za-z])_([0-9])", r"\1\2", expr)
    vals={}
    for k,v in (plan.get("values") or {}).items():
        nv=_num(v)
        if nv is not None: vals[str(k).strip().replace(" ","").replace("_","")]=nv
    if not expr or not _SAFE.match(expr): base["reason"]="bad_expr"; return base
    lower={k.lower():v for k,v in vals.items()}
    subs={}
    for t in set(_IDENT.findall(expr)):
        if t.lower() in lower: subs[t]=lower[t.lower()]
        else: base["reason"]="missing:"+t; return base
    try:
        e=smp.sympify(expr); base["value"]=float(e.subs({smp.Symbol(k):v for k,v in subs.items()})); base["abstain"]=False
    except Exception as ex:
        base["reason"]="sympy:"+str(ex)[:30]
    return base

# numeric resistor problems: ONE generation -> FINAL (model's arithmetic) vs PLAN->sympy (tool)
OFFLOAD_N=15   # numeric resistor gens (tuned only); raise for firmer numbers
rows=[r for r in (val_rows or []) if not r.get("abstain")
      and r.get("value_mode")=="numeric" and r.get("family")=="dc_resistor"][:OFFLOAD_N]
print(f"tool-offload on {len(rows)} numeric resistor problems; first 6:")
raw_ok=off_ok=have_plan=n=0
for i,r in enumerate(rows,1):
    gold=(r.get("gold_answer") or {}).get("value")
    if gold is None: continue
    n+=1
    txt=solve_image(load_image(r["image"]), r["question"], model, tokenizer)   # standard prompt (trained to emit FINAL + PLAN)
    raw=parse_final(txt)["value"]; rk=bool(_close(raw, gold, 0.03)); raw_ok+=rk
    plan=parse_plan(txt); have_plan+=bool(plan)
    comp=tool_compute(plan); o=bool((not comp["abstain"]) and _close(comp["value"], gold, 0.03)); off_ok+=o
    if i<=6: print(f"  [{i}] gold={gold} | raw={raw} ({'OK' if rk else 'x'}) | tool={comp.get('value')} ({'OK' if o else comp.get('reason','x')})")
if n:
    print(f"\nnumeric resistor answer accuracy (n={n}):")
    print(f"  raw  (model does the arithmetic) : {raw_ok/n:.3f}")
    print(f"  tool (sympy does the arithmetic) : {off_ok/n:.3f}")
if have_plan==0:
    print("\n[!] No PLAN emitted -> this model was NOT trained with INCLUDE_PLAN=True. "
          "Regenerate the dataset with INCLUDE_PLAN=True, retrain, then re-run this cell.")
